<a href="https://colab.research.google.com/github/Uwais-ur-Rehman/ML-Project/blob/main/Column_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv('/content/covid_toy.csv')

In [ ]:
df

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No
...,...,...,...,...,...,...
95,12,Female,104.0,Mild,Bangalore,No
96,51,Female,101.0,Strong,Kolkata,Yes
97,20,Female,101.0,Mild,Bangalore,No
98,5,Female,98.0,Strong,Mumbai,No


In [ ]:
df['cough'].value_counts()

,count
cough,
Mild,62
Strong,38


In [ ]:
df['city'].value_counts()

,count
city,
Kolkata,32
Bangalore,30
Delhi,22
Mumbai,16


In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2,random_state=42)

In [ ]:
X_train

,age,gender,fever,cough,city
55,81,Female,101.0,Mild,Mumbai
88,5,Female,100.0,Mild,Kolkata
26,19,Female,100.0,Mild,Kolkata
42,27,Male,100.0,Mild,Delhi
69,73,Female,103.0,Mild,Delhi
...,...,...,...,...,...
60,24,Female,102.0,Strong,Bangalore
71,75,Female,104.0,Strong,Delhi
14,51,Male,104.0,Mild,Bangalore
92,82,Female,102.0,Strong,Kolkata


In [ ]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
fever_imputer=SimpleImputer()
fever_imputer=SimpleImputer(strategy='mean')
df['fever']=fever_imputer.fit_transform(df[['fever']])

In [ ]:
oe=OrdinalEncoder(categories=[['Mild','Strong']])
df['cough']=oe.fit_transform(df[['cough']])
ohe=OneHotEncoder(drop='first',sparse_output=False)
city_encoded=ohe.fit_transform(df[['city']])

In [ ]:
city_encoded

array([[0., 1., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 1.],
       [0., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [0., 0., 1.],
       [0., 1., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 1., 0.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 0., 0.],
       [1., 0., 0.],
       [0., 0., 0.],
       [0., 0., 1.],
       [0., 0

In [ ]:
from sklearn.compose import ColumnTransformer

In [ ]:
transformer= ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder="passthrough")

In [ ]:
X_train=transformer.fit_transform(X_train)
X_test=transformer.transform(X_test)


In [ ]:
X_train = pd.DataFrame(X_train,columns=transformer.get_feature_names_out())
X_train

,tnf1__fever,tnf2__cough,tnf3__gender_Male,tnf3__city_Delhi,tnf3__city_Kolkata,tnf3__city_Mumbai,remainder__age
0,101.0,0.0,0.0,0.0,0.0,1.0,81.0
1,100.0,0.0,0.0,0.0,1.0,0.0,5.0
2,100.0,0.0,0.0,0.0,1.0,0.0,19.0
3,100.0,0.0,1.0,1.0,0.0,0.0,27.0
4,103.0,0.0,0.0,1.0,0.0,0.0,73.0
...,...,...,...,...,...,...,...
75,102.0,1.0,0.0,0.0,0.0,0.0,24.0
76,104.0,1.0,0.0,1.0,0.0,0.0,75.0
77,104.0,0.0,1.0,0.0,0.0,0.0,51.0
78,102.0,1.0,0.0,0.0,1.0,0.0,82.0


In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

sfs = SequentialFeatureSelector(
    model,
    n_features_to_select=3,
    direction="forward"
)

sfs.fit(X_train, y_train)

selected_features = X_train.columns[sfs.get_support()]

print(selected_features)

Index(['tnf3__gender_Male', 'tnf3__city_Kolkata', 'remainder__age'], dtype='object')
